# 🎓 Proyecto: Detección de Fraude en Comprobantes Nequi con Redes Neuronales Dual-Branch
## Asignatura: Inteligencia Artificial Avanzada | Metodología: Aprendizaje Basado en Retos (ABR)
---
### 📌 1. Identificación y Justificación del Problema
El fraude con comprobantes digitales de Nequi presenta **dos desafíos principales**:
1. **Aplicaciones Falsas / Clonadas ("Nequi Fake"):** Generan comprobantes con diseño incorrecto (cabecera morada, sin código QR dinámico, sellos apócrifos).
2. **Manipulación Digital (Photoshop / Canva):** Toman un comprobante legítimo y modifican el valor de `¿Cuánto?` o la `Fecha`.

**Solución de IA: Red Neuronal Dual-Branch (Fusión RGB + Forense ELA)**
* **Rama Visual (RGB):** Aprende la fisonomía exacta del comprobante oficial (código QR con marco verde menta, ilustraciones de fondo tipo tiquete, sellos de la Superfinanciera).
* **Rama Forense (ELA):** Analiza los artefactos de compresión JPEG para detectar modificaciones y sobreescrituras en los montos.

In [ ]:
# ====================================================================
# 0. CONFIGURACIÓN DEL ENTORNO Y LIBRERÍAS
# ====================================================================
import os
import random
import glob
import shutil
from io import BytesIO
from datetime import datetime

import numpy as np
from PIL import Image, ImageDraw, ImageFont, ImageChops, ImageEnhance
import matplotlib.pyplot as plt
import seaborn as sns

import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms, models

# Semillas para reproducibilidad
random.seed(42)
np.random.seed(42)
torch.manual_seed(42)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(42)

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"✓ Entorno preparado. Aceleración: {device}")

# Descargar o preparar las plantillas base oficiales
if not os.path.exists("templates/comprobante_autentico.jpg"):
    !git clone https://github.com/efelipe0526/deteccion-de-fraude-nequi.git repo_tmp
    !cp -r repo_tmp/templates . 2>/dev/null || true
    !rm -rf repo_tmp

print("✓ Plantillas oficiales sincronizadas.")

--- 
### 🧪 2. Generador Multimodal Aumentado con Plantillas Oficiales

In [ ]:
# Cargar plantillas base
ruta_aut = "templates/comprobante_autentico.jpg"
ruta_fal = "templates/comprobante_falso_app.png"

if os.path.exists(ruta_aut):
    base_autentica = Image.open(ruta_aut).convert("RGB")
else:
    # Fallback si no está el archivo
    base_autentica = Image.new("RGB", (480, 880), color=(255, 255, 255))

if os.path.exists(ruta_fal):
    base_falsa = Image.open(ruta_fal).convert("RGB")
else:
    base_falsa = Image.new("RGB", (480, 880), color=(98, 42, 115))

def aumentacion_realista(img_pil):
    img = img_pil.copy()
    # Variación de brillo y contraste simulando diferentes pantallas
    img = ImageEnhance.Brightness(img).enhance(random.uniform(0.92, 1.08))
    img = ImageEnhance.Contrast(img).enhance(random.uniform(0.92, 1.08))
    # Rotación leve (-1.5 a 1.5 grados)
    if random.random() > 0.4:
        img = img.rotate(random.uniform(-1.2, 1.2), resample=Image.BILINEAR, expand=False, fillcolor=(255, 255, 255))
    # Compresión multicalidad de WhatsApp / Redes sociales
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=random.choice([55, 68, 78, 85, 92, 98]))
    buf.seek(0)
    return Image.open(buf)

def simular_edicion_monto(img_aut_pil):
    img = img_aut_pil.copy()
    draw = ImageDraw.Draw(img)
    w, h = img.size
    # Parche en la zona de monto
    y1, y2 = int(h * 0.57), int(h * 0.63)
    x1, x2 = int(w * 0.10), int(w * 0.85)
    draw.rectangle([(x1, y1), (x2, y2)], fill=random.choice([(255, 255, 255), (245, 248, 250)]))
    draw.text((x1 + 10, y1 + 5), f"$ {random.randint(5, 50)*50000:,.2f}", fill=(15, 15, 20))
    buf = BytesIO()
    img.save(buf, format="JPEG", quality=60)
    buf.seek(0)
    return Image.open(buf)

# Construir Dataset
for split, n_total in [("train", 400), ("val", 80), ("test", 80)]:
    for cls in ["legitimo", "fraude"]:
        folder = f"dataset_dual/{split}/{cls}"
        os.makedirs(folder, exist_ok=True)
        for i in range(n_total // 2):
            if cls == "legitimo":
                img_gen = aumentacion_realista(base_autentica)
            else:
                if random.random() > 0.5:
                    img_gen = aumentacion_realista(base_falsa) # App falsa
                else:
                    img_gen = simular_edicion_monto(base_autentica) # Edición digital
            img_gen.save(f"{folder}/{cls}_{i+1:04d}.jpg", quality=85)

print("✓ Dataset Dual generado con datos aumentados de alta fidelidad.")

--- 
### 🔬 3. Módulo Forense: Error Level Analysis (ELA)

In [ ]:
def calcular_ela(img_pil, calidad=90, escala=15):
    img_rgb = img_pil.convert("RGB")
    buf = BytesIO()
    img_rgb.save(buf, format="JPEG", quality=calidad)
    buf.seek(0)
    recomprimida = Image.open(buf)
    
    dif = ImageChops.difference(img_rgb, recomprimida)
    max_dif = max([ex[1] for ex in dif.getextrema()]) or 1
    factor = escala * (255.0 / max_dif)
    return ImageEnhance.Brightness(dif).enhance(factor)

# Visualizar muestras del dataset
img_leg = base_autentica
img_fal_app = base_falsa
img_edit = simular_edicion_monto(base_autentica)
ela_edit = calcular_ela(img_edit)

fig, axs = plt.subplots(1, 3, figsize=(15, 5))
axs[0].imshow(img_leg); axs[0].set_title("1. Comprobante Oficial Nequi (Auténtico)"); axs[0].axis("off")
axs[1].imshow(img_fal_app); axs[1].set_title("2. App Falsa / Clonada (Fraude)"); axs[1].axis("off")
axs[2].imshow(ela_edit); axs[2].set_title("3. Forense ELA (Edición Digital)"); axs[2].axis("off")
plt.tight_layout()
plt.show()

--- 
### 🧠 4. Arquitectura de Red Neuronal Dual-Branch (Visual RGB + Forense ELA)

In [ ]:
class NequiDualDataset(Dataset):
    def __init__(self, root_dir, split="train", transform=None):
        self.samples = []
        self.transform = transform
        for f in glob.glob(f"{root_dir}/{split}/legitimo/*.jpg"):
            self.samples.append((f, 0.0))
        for f in glob.glob(f"{root_dir}/{split}/fraude/*.jpg"):
            self.samples.append((f, 1.0))
            
    def __len__(self):
        return len(self.samples)
        
    def __getitem__(self, idx):
        path, label = self.samples[idx]
        img_rgb = Image.open(path).convert("RGB")
        img_ela = calcular_ela(img_rgb)
        
        if self.transform:
            x_rgb = self.transform(img_rgb)
            x_ela = self.transform(img_ela)
        else:
            t = transforms.ToTensor()
            x_rgb, x_ela = t(img_rgb), t(img_ela)
            
        return x_rgb, x_ela, torch.tensor(label, dtype=torch.float32)

transform_pipeline = transforms.Compose([
    transforms.Resize((224, 224)),
    transforms.ToTensor(),
    transforms.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225])
])

train_loader = DataLoader(NequiDualDataset("dataset_dual", "train", transform_pipeline), batch_size=16, shuffle=True)
val_loader = DataLoader(NequiDualDataset("dataset_dual", "val", transform_pipeline), batch_size=16, shuffle=False)
test_loader = DataLoader(NequiDualDataset("dataset_dual", "test", transform_pipeline), batch_size=16, shuffle=False)

class NequiDualBranchCNN(nn.Module):
    def __init__(self):
        super(NequiDualBranchCNN, self).__init__()
        weights = models.MobileNet_V3_Small_Weights.DEFAULT
        
        # Rama 1: Visual RGB
        base_rgb = models.mobilenet_v3_small(weights=weights)
        self.branch_rgb = base_rgb.features
        self.pool_rgb = nn.AdaptiveAvgPool2d((1, 1))
        
        # Rama 2: Forense ELA
        base_ela = models.mobilenet_v3_small(weights=weights)
        self.branch_ela = base_ela.features
        self.pool_ela = nn.AdaptiveAvgPool2d((1, 1))
        
        # Clasificador de Fusión Lineal (1152-D)
        self.classifier = nn.Sequential(
            nn.Linear(576 * 2, 256),
            nn.ReLU(inplace=True),
            nn.Dropout(p=0.3),
            nn.Linear(256, 64),
            nn.ReLU(inplace=True),
            nn.Linear(64, 1)
        )
        
    def forward(self, x_rgb, x_ela):
        f_rgb = self.pool_rgb(self.branch_rgb(x_rgb)).flatten(1)
        f_ela = self.pool_ela(self.branch_ela(x_ela)).flatten(1)
        fused = torch.cat([f_rgb, f_ela], dim=1)
        return self.classifier(fused)

modelo_dual = NequiDualBranchCNN().to(device)
print("✓ Arquitectura compilada.")

--- 
### 🚀 5. Entrenamiento y Validación del Modelo

In [ ]:
criterio = nn.BCEWithLogitsLoss()
optimizador = torch.optim.AdamW(modelo_dual.parameters(), lr=0.001, weight_decay=1e-4)
epochs = 8

for epoch in range(epochs):
    modelo_dual.train()
    t_loss, t_corr, total = 0.0, 0, 0
    for x_rgb, x_ela, labels in train_loader:
        x_rgb, x_ela, labels = x_rgb.to(device), x_ela.to(device), labels.to(device).unsqueeze(1)
        optimizador.zero_grad()
        outs = modelo_dual(x_rgb, x_ela)
        loss = criterio(outs, labels)
        loss.backward(); optimizador.step()
        t_loss += loss.item() * x_rgb.size(0)
        preds = (torch.sigmoid(outs) >= 0.5).float()
        t_corr += (preds == labels).sum().item()
        total += labels.size(0)
    print(f"Época [{epoch+1:02d}/{epochs:02d}] - Loss: {t_loss/total:.4f} - Accuracy: {t_corr/total*100:.1f}%")

modelo_dual.eval()
torch.save(modelo_dual.state_dict(), "mejor_modelo_dual_nequi.pth")
print("✓ Modelo Dual-Branch entrenado con éxito.")

--- 
### 📲 6. Módulo de Prueba Interactiva (Sube tu Comprobante Real)

In [ ]:
from google.colab import files

modelo_dual.eval()
print("📤 Sube cualquier comprobante para evaluarlo:")
uploaded = files.upload()

for filename in uploaded.keys():
    print(f"\n" + "="*60)
    print(f"🔍 Analizando comprobante: {filename}")
    print("="*60)
    
    img_pil = Image.open(filename).convert("RGB")
    img_ela = calcular_ela(img_pil)
    
    # Entradas normalizadas para ambas ramas
    x_rgb = transform_pipeline(img_pil).unsqueeze(0).to(device)
    x_ela = transform_pipeline(img_ela).unsqueeze(0).to(device)
    
    with torch.no_grad():
        logits = modelo_dual(x_rgb, x_ela)
        prob_fraude = torch.sigmoid(logits).item()
        
    es_fraude = prob_fraude >= 0.5
    confianza = prob_fraude if es_fraude else (1.0 - prob_fraude)
    
    # Gráfica del Diagnóstico
    plt.figure(figsize=(10, 5))
    plt.subplot(1, 2, 1)
    plt.imshow(img_pil)
    plt.title("Comprobante Subido"); plt.axis("off")
    
    plt.subplot(1, 2, 2)
    plt.imshow(img_ela)
    color_t = "red" if es_fraude else "green"
    titulo = f"🚨 DICTAMEN: FRAUDE ({confianza*100:.1f}%)" if es_fraude else f"✅ DICTAMEN: AUTÉNTICO ({confianza*100:.1f}%)"
    plt.title(titulo, color=color_t, fontweight="bold"); plt.axis("off")
    plt.tight_layout()
    plt.show()
    
    print("\n📋 RESULTADO DEL ANÁLISIS FORENSE DUAL:")
    if es_fraude:
        print(f"  🚨 RESULTADO: [FRAUDE DETECTADO]")
        print(f"  ⚠️ Probabilidad de Fraude: {prob_fraude*100:.2f}%")
        print("  ⚠️ Diagnóstico: Diseño apócrifo/App falsa o alteración digital en el monto/fecha.")
    else:
        print(f"  ✅ RESULTADO: [COMPROBANTE AUTÉNTICO]")
        print(f"  🛡️ Probabilidad de Autenticidad: {(1-prob_fraude)*100:.2f}%")
        print("  🛡️ Diagnóstico: Código QR oficial, diseño tiquete y textura de compresión válidos.")
    print("="*60)